# 03_Futures_Database 数据湖契约化读取 Demo

17 张稳定 silver 表（7 张维度表、10 张事实表）分别由独立代码单元格读取；大表示例先应用 Hive 分区过滤。生意社原始页面归档不是 silver 表，在末尾用独立示例只读字节并核对 SHA-256，不解析 HTML。实验性 gold 输出由所属下游工作流自行演示。

In [ ]:
from __future__ import annotations

import hashlib
import pathlib
import sys

project_markers = [".git", ".env", "config/settings.py"]
current_path = pathlib.Path.cwd().resolve()
for candidate_root in [current_path, *current_path.parents]:
    if all((candidate_root / marker).exists() for marker in project_markers):
        sys.path.insert(0, str(candidate_root))
        break
else:
    raise RuntimeError("未找到项目根目录")

import pyarrow as pa
import pyarrow.dataset as ds

from config.data_contracts import (
    EXTERNAL_INDEX_DAILY_SCHEMA,
    EXTERNAL_MARKET_CALENDAR_SCHEMA,
    FUTURES_BAR_CALENDAR_SCHEMA,
    FUTURES_CONTRACT_CALENDAR_SCHEMA,
    FUTURES_DAILY_SCHEMA,
    FUTURES_EXCHANGE_REPORT_CALENDAR_SCHEMA,
    FUTURES_MEMBER_POSITION_DAILY_SCHEMA,
    FUTURES_MINUTE_SCHEMA,
    FUTURES_MISSING_BAR_SCHEMA,
    FUTURES_POSITION_RANK_DAILY_SCHEMA,
    FUTURES_VARIETY_CALENDAR_SCHEMA,
    FUTURES_WAREHOUSE_RECEIPT_DAILY_SCHEMA,
    INTEREST_RATE_DAILY_SCHEMA,
    MACRO_RELEASE_CALENDAR_SCHEMA,
    MACRO_RELEASE_SCHEMA,
    OVERSEAS_FUTURES_DAILY_SCHEMA,
    TRADE_CALENDAR_SCHEMA,
    arrow_to_pandas,
    validate_arrow_table,
)
from config.settings import settings

LAKE_ROOT = settings.futures_lake_root


def read_contract_dataset(table_name, layer, schema, partition_columns, filter_expression):
    table_path = LAKE_ROOT / layer / table_name
    partitioning = ds.partitioning(
        pa.schema([schema.field(name) for name in partition_columns]),
        flavor="hive",
    )
    dataset = ds.dataset(table_path, format="parquet", partitioning=partitioning)
    table = dataset.to_table(filter=filter_expression)
    return arrow_to_pandas(validate_arrow_table(table, schema), schema)


## 1. `dim_trade_calendar`

In [ ]:
trade_calendar_df = read_contract_dataset(
    "dim_trade_calendar",
    "silver",
    TRADE_CALENDAR_SCHEMA,
    ["year"],
    (ds.field('year') == 2024),
)
trade_calendar_df.head()


## 2. `dim_futures_variety_calendar`

由 `get_all_securities(["futures"], date=None)` 的完整固定月份合约目录生成；`config/futures_fact_collection_policy.py` 的事实采集白名单不裁剪本表。

In [ ]:
futures_variety_calendar_df = read_contract_dataset(
    "dim_futures_variety_calendar",
    "silver",
    FUTURES_VARIETY_CALENDAR_SCHEMA,
    ["exchange_code","year","month"],
    (ds.field('exchange_code') == 'XSGE') & (ds.field('year') == 2024) & (ds.field('month') == 1),
)
futures_variety_calendar_df.head()


## 3. `dim_futures_contract_calendar`

继承完整品种宇宙并枚举全部固定月份合约 Session；`config/futures_fact_collection_policy.py` 的事实采集白名单不裁剪本表。

In [ ]:
futures_contract_calendar_df = read_contract_dataset(
    "dim_futures_contract_calendar",
    "silver",
    FUTURES_CONTRACT_CALENDAR_SCHEMA,
    ["exchange_code","year","month"],
    (ds.field('exchange_code') == 'XSGE') & (ds.field('year') == 2024) & (ds.field('month') == 1),
)
futures_contract_calendar_df.head()


## 4. `dim_futures_bar_calendar`

完整保留 1d/1m 理论格点；`is_fetch_required` 只选择事实任务：c05、c06 分别在完整日线和分钟格点上应用 `config/futures_fact_collection_policy.py` 的期货事实采集白名单。

In [ ]:
futures_bar_calendar_df = read_contract_dataset(
    "dim_futures_bar_calendar",
    "silver",
    FUTURES_BAR_CALENDAR_SCHEMA,
    ["bar_frequency","exchange_code","year","month"],
    (ds.field('bar_frequency') == '1m') & (ds.field('exchange_code') == 'XSGE') & (ds.field('year') == 2024) & (ds.field('month') == 1),
)
futures_bar_calendar_df.head()


## 5. `fact_futures_daily`

In [ ]:
futures_daily_df = read_contract_dataset(
    "fact_futures_daily",
    "silver",
    FUTURES_DAILY_SCHEMA,
    ["exchange_code","underlying_code","year","month"],
    (ds.field('exchange_code') == 'XSGE') & (ds.field('underlying_code') == 'CU') & (ds.field('year') == 2024) & (ds.field('month') == 1),
)
futures_daily_df.head()


## 6. `fact_futures_minute`

In [ ]:
futures_minute_df = read_contract_dataset(
    "fact_futures_minute",
    "silver",
    FUTURES_MINUTE_SCHEMA,
    ["exchange_code","underlying_code","year","month"],
    (ds.field('exchange_code') == 'XSGE') & (ds.field('underlying_code') == 'CU') & (ds.field('year') == 2024) & (ds.field('month') == 1),
)
futures_minute_df.head()


## 7. `fact_futures_missing_bar`

In [ ]:
futures_missing_bar_df = read_contract_dataset(
    "fact_futures_missing_bar",
    "silver",
    FUTURES_MISSING_BAR_SCHEMA,
    ["bar_frequency","exchange_code","underlying_code","year","month"],
    (ds.field('bar_frequency') == '1m') & (ds.field('exchange_code') == 'XSGE') & (ds.field('underlying_code') == 'CU') & (ds.field('year') == 2024) & (ds.field('month') == 1),
)
futures_missing_bar_df.head()


## 8. `dim_futures_exchange_report_calendar`

完整保留三类品种报告格点；`is_fetch_required` 由 `config/futures_fact_collection_policy.py` 的期货事实采集白名单与具体 API 覆盖规则共同决定。

In [ ]:
futures_exchange_report_calendar_df = read_contract_dataset(
    "dim_futures_exchange_report_calendar",
    "silver",
    FUTURES_EXCHANGE_REPORT_CALENDAR_SCHEMA,
    ["dataset_name","exchange_code","year","month"],
    (ds.field('dataset_name') == 'position_rank') & (ds.field('exchange_code') == 'XSGE') & (ds.field('year') == 2024) & (ds.field('month') == 1),
)
futures_exchange_report_calendar_df.head()


## 9. `fact_futures_position_rank_daily`

In [ ]:
futures_position_rank_daily_df = read_contract_dataset(
    "fact_futures_position_rank_daily",
    "silver",
    FUTURES_POSITION_RANK_DAILY_SCHEMA,
    ["exchange_code","underlying_code","year","month"],
    (ds.field('exchange_code') == 'XSGE') & (ds.field('underlying_code') == 'CU') & (ds.field('year') == 2024) & (ds.field('month') == 1),
)
futures_position_rank_daily_df.head()


## 10. `fact_futures_member_position_daily`

In [ ]:
futures_member_position_daily_df = read_contract_dataset(
    "fact_futures_member_position_daily",
    "silver",
    FUTURES_MEMBER_POSITION_DAILY_SCHEMA,
    ["exchange_code","underlying_code","year","month"],
    (ds.field('exchange_code') == 'XSGE') & (ds.field('underlying_code') == 'CU') & (ds.field('year') == 2024) & (ds.field('month') == 1),
)
futures_member_position_daily_df.head()


## 11. `fact_futures_warehouse_receipt_daily`

In [ ]:
futures_warehouse_receipt_daily_df = read_contract_dataset(
    "fact_futures_warehouse_receipt_daily",
    "silver",
    FUTURES_WAREHOUSE_RECEIPT_DAILY_SCHEMA,
    ["exchange_code","underlying_code","year","month"],
    (ds.field('exchange_code') == 'XSGE') & (ds.field('underlying_code') == 'CU') & (ds.field('year') == 2024) & (ds.field('month') == 1),
)
futures_warehouse_receipt_daily_df.head()


## 12. `dim_external_market_calendar`

In [ ]:
external_market_calendar_df = read_contract_dataset(
    "dim_external_market_calendar",
    "silver",
    EXTERNAL_MARKET_CALENDAR_SCHEMA,
    ["dataset_name","year","month"],
    (ds.field('dataset_name') == 'external_index') & (ds.field('year') == 2024) & (ds.field('month') == 1),
)
external_market_calendar_df.head()


## 13. `fact_overseas_futures_daily`

In [ ]:
overseas_futures_daily_df = read_contract_dataset(
    "fact_overseas_futures_daily",
    "silver",
    OVERSEAS_FUTURES_DAILY_SCHEMA,
    ["year","month"],
    (ds.field('year') == 2024) & (ds.field('month') == 1),
)
overseas_futures_daily_df.head()


## 14. `fact_external_index_daily`

In [ ]:
external_index_daily_df = read_contract_dataset(
    "fact_external_index_daily",
    "silver",
    EXTERNAL_INDEX_DAILY_SCHEMA,
    ["index_category","year","month"],
    (ds.field('index_category') == 'shipping') & (ds.field('year') == 2024) & (ds.field('month') == 1),
)
external_index_daily_df.head()


## 15. `dim_macro_release_calendar`

In [ ]:
macro_release_calendar_df = read_contract_dataset(
    "dim_macro_release_calendar",
    "silver",
    MACRO_RELEASE_CALENDAR_SCHEMA,
    ["dataset_name","year","month"],
    (ds.field('dataset_name') == 'interest_rate') & (ds.field('year') == 2024) & (ds.field('month') == 1),
)
macro_release_calendar_df.head()


## 16. `fact_interest_rate_daily`

In [ ]:
interest_rate_daily_df = read_contract_dataset(
    "fact_interest_rate_daily",
    "silver",
    INTEREST_RATE_DAILY_SCHEMA,
    ["year","month"],
    (ds.field('year') == 2024) & (ds.field('month') == 1),
)
interest_rate_daily_df.head()


## 17. `fact_macro_release`

In [ ]:
macro_release_df = read_contract_dataset(
    "fact_macro_release",
    "silver",
    MACRO_RELEASE_SCHEMA,
    ["year","month"],
    (ds.field('year') == 2024) & (ds.field('month') == 1),
)
macro_release_df.head()


## 生意社原始页面归档（非 silver）

该链路只归档 HTTP 原始响应和 SHA-256 sidecar，不属于上述 17 张稳定 silver 表，也没有 Arrow Schema。下面按一个日期独立读取正式 raw 文件，以二进制重新计算摘要并核对 sidecar；示例不会解码或解析 HTML。

In [ ]:
RAW_OBSERVATION_DATE = "2024-01-02"
raw_year, raw_month, _ = RAW_OBSERVATION_DATE.split("-")

raw_date_path = (
    LAKE_ROOT
    / "raw"
    / "100ppi"
    / "domestic_spot_basis"
    / f"year={raw_year}"
    / f"month={int(raw_month):02d}"
    / f"observation_date={RAW_OBSERVATION_DATE}"
)
response_path = raw_date_path / "response.html"
sha256_path = raw_date_path / "response.sha256"

# response.html 始终按原始字节读取；不要为了检查摘要而解码或解析页面。
response_bytes = response_path.read_bytes()
recorded_sha256 = sha256_path.read_text(encoding="utf-8").removesuffix("\n")
actual_sha256 = hashlib.sha256(response_bytes).hexdigest()

if (
    len(recorded_sha256) != 64
    or recorded_sha256 != recorded_sha256.lower()
    or any(character not in "0123456789abcdef" for character in recorded_sha256)
):
    raise ValueError("response.sha256 必须是可带结尾换行的 UTF-8 小写 64 位十六进制摘要。")
if actual_sha256 != recorded_sha256:
    raise ValueError("生意社原始响应与 response.sha256 不一致。")

{
    "response_path": response_path,
    "byte_count": len(response_bytes),
    "sha256": actual_sha256,
}
